In [1]:
import json
import os
import torch
import sqlglot
from sqlglot.expressions import Table
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/hyeonjin/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. 경로 설정 (사용자 디렉토리 구조 반영)
BASE_DIR = "../data/BIRD_dev"
DEV_JSON_PATH = os.path.join(BASE_DIR, "dev.json")
DEV_TABLES_PATH = os.path.join(BASE_DIR, "dev_tables.json")

In [3]:
# 2. 데이터 로드
with open(DEV_JSON_PATH, 'r', encoding='utf-8') as f:
    dev_data = json.load(f)

with open(DEV_TABLES_PATH, 'r', encoding='utf-8') as f:
    tables_data = json.load(f)

db_schema_map = {item['db_id']: item for item in tables_data}

In [4]:
# 3. Gold Table 추출기 (sqlglot 활용)
def extract_gold_tables(sql_query):
    try:
        parsed = sqlglot.parse_one(sql_query, read="sqlite")
        return set(node.name.lower() for node in parsed.find_all(Table))
    except Exception:
        return set()

In [5]:
# 4. 스키마 프롬프트 생성기 (table.column 포맷)
def build_schema_dot_format(db_id, used_tables=None):
    schema_info = db_schema_map.get(db_id)
    if not schema_info: return ""
    
    table_names = schema_info['table_names_original']
    column_names = schema_info['column_names_original']
    
    schema_lines = []
    # column_names 구조: [[table_idx, column_name], ...]
    for table_idx, col_name in column_names:
        if table_idx < 0: continue # 인덱스 -1은 '*' 이므로 제외
        
        t_name = table_names[table_idx]
        
        # Gold Schema 모드: 추출된 정답 테이블에 없으면 필터링
        if used_tables is not None and t_name.lower() not in used_tables:
            continue
            
        schema_lines.append(f"{t_name}.{col_name}")
        
    return "\n".join(schema_lines)

In [6]:
# 5. 프롬프트 템플릿 (Llama-3.1 Instruct에 최적화된 System/User 분리)
def create_messages(schema_str, question, evidence):
    system_prompt = (
        "You are an expert SQL developer. Your task is to write a SQLite query based on the given schema and external knowledge. "
        "IMPORTANT: If a column name contains spaces or special characters, you MUST wrap it in backticks (e.g., `Column Name` or `Percent (%)`). "
        "Output ONLY the SQL query. Do not include markdown formatting like ```sql or any explanations."
    )
    user_prompt = f"### Schema (table.column):\n{schema_str}\n\n### External Knowledge:\n{evidence}\n\n### Question:\n{question}\n\n### SQL:"
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

In [7]:
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

print("Loading tokenizer and model (This is safe for memory)...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# device_map="auto"가 가용한 여러 GPU에 모델을 안전하게 분산 배치합니다.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True # 시스템 RAM 스파이크 방지 핵심 옵션
)

# 패딩 토큰 설정 (Llama 3 특성)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

Loading tokenizer and model (This is safe for memory)...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.23s/it]


In [8]:
def generate_sql(messages):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.0,      # Greedy Decoding
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    
    # 입력 프롬프트 부분을 제외하고 생성된 텍스트만 추출
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

In [10]:
import threading
import sqlite3

DB_DIR = os.path.join(BASE_DIR, "dev_databases")

def execute_sql_on_db(db_path, sql_query, timeout_sec=3.0):
    """
    무한 루프(Cross Join) 방지를 위해 Thread를 이용한 엄격한 타임아웃을 적용합니다.
    """
    result = [None]
    exception = [None]

    def target():
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute(sql_query)
            result[0] = set(cursor.fetchall())
            conn.close()
        except Exception as e:
            exception[0] = str(e)

    thread = threading.Thread(target=target)
    thread.start()
    thread.join(timeout_sec)

    if thread.is_alive():
        return "Error: Query Timeout (Possible Cross Join)"
    
    if exception[0]:
        return exception[0]
        
    return result[0]

In [13]:
from tqdm import tqdm

print("\nStarting inference and EX evaluation for 10 samples...")

metrics = {
    "full_correct": 0,
    "gold_correct": 0,
    "total": 0
}

test_num = 1534

for i, item in tqdm(enumerate(dev_data[:test_num]), total=len(dev_data[:test_num])): # 실험 규모에 맞게 조절 (예: 50개)
    db_id = item['db_id']
    question = item['question']
    evidence = item['evidence']
    gold_sql = item['SQL']
    
    db_path = os.path.join(DB_DIR, db_id, f"{db_id}.sqlite")
    
    # --- 전처리 및 프롬프트 구성 ---
    gold_tables = extract_gold_tables(gold_sql)
    schema_full = build_schema_dot_format(db_id, used_tables=None)
    schema_gold = build_schema_dot_format(db_id, used_tables=gold_tables)
    
    msg_full = create_messages(schema_full, question, evidence)
    msg_gold = create_messages(schema_gold, question, evidence)
    
    # --- 모델 추론 ---
    pred_full = generate_sql(msg_full)
    pred_gold = generate_sql(msg_gold)
    
    # --- EX (Execution Accuracy) 평가 ---
    # 1. Gold SQL 실행 (Ground Truth 결과 확보)
    gold_result = execute_sql_on_db(db_path, gold_sql)
    
    # 2. Predicted SQL 실행
    pred_full_result = execute_sql_on_db(db_path, pred_full)
    pred_gold_result = execute_sql_on_db(db_path, pred_gold)
    
    # 3. 결과 비교 (Gold 결과가 정상적으로 나왔을 때만 평가)
    is_full_correct = (gold_result == pred_full_result) and not isinstance(gold_result, str)
    is_gold_correct = (gold_result == pred_gold_result) and not isinstance(gold_result, str)
    
    if is_full_correct: metrics["full_correct"] += 1
    if is_gold_correct: metrics["gold_correct"] += 1
    metrics["total"] += 1
    
    """
    # --- 결과 로깅 ---
    print(f"\n[{i+1}/{len(dev_data[:test_num])}] DB: {db_id}")
    print(f"Question: {question}") # 필요시 주석 해제
    
    # 오답 분석을 위한 로깅 (틀렸을 때만 SQL 출력)
    
    if not is_full_correct:
        print(f"❌ [Full Failed] Pred: {pred_full}")
        if isinstance(pred_full_result, str): print(f"   Error: {pred_full_result}")
            
    if not is_gold_correct:
        print(f"❌ [Gold Failed] Pred: {pred_gold}")
        if isinstance(pred_gold_result, str): print(f"   Error: {pred_gold_result}")
            
    if is_full_correct and is_gold_correct:
        print("✅ Both Correct")
    """

# --- 최종 결과 산출 ---
ex_full = (metrics["full_correct"] / metrics["total"]) * 100
ex_gold = (metrics["gold_correct"] / metrics["total"]) * 100
delta_ex = ex_gold - ex_full

print("\n" + "="*40)
print("🎯 EXPERIMENT RESULTS (Execution Accuracy)")
print("="*40)
print(f"Total Samples Tested: {metrics['total']}")
print(f"EX w/ Full Schema (Baseline) : {ex_full:.1f}%")
print(f"EX w/ Gold Schema (Oracle)   : {ex_gold:.1f}%")
print(f"Performance Gap ($\\Delta EX$)  : +{delta_ex:.1f}%p")
print("="*40)


Starting inference and EX evaluation for 10 samples...


100%|██████████| 1534/1534 [1:31:13<00:00,  3.57s/it]  


🎯 EXPERIMENT RESULTS (Execution Accuracy)
Total Samples Tested: 1534
EX w/ Full Schema (Baseline) : 34.1%
EX w/ Gold Schema (Oracle)   : 40.1%
Performance Gap ($\Delta EX$)  : +6.0%p


In [2]:
import sqlglot
from sqlglot.expressions import Table, Column
from typing import Set, Tuple, List
import pandas as pd

In [3]:
def parse_sql_elements(sql: str) -> Tuple[Set[str], Set[str]]:
    """SQL을 파싱하여 명시된 Table 명과 Column 명 집합을 반환"""

    if not sql:
        return set(), set()
    
    try:
        parsed = sqlglot.parse_one(sql, read="sqlite")
        tables = set(node.name.lower() for node in parsed.find_all(Table) if node.name)
        columns = set(node.name.lower() for node in parsed.find_all(Column) if node.name)
        return tables, columns

    except Exception as e:
        return set(), set()

def calculate_schema_metrics(pred_elements: Set[str], gold_elements: Set[str]) -> Tuple[float, float, List[str], List[str]]:
    """Recall, Precision 및 Missing, Extra 요소를 계산합니다."""
    if not gold_elements and not pred_elements:
        return 1.0, 1.0, [], []
    
    intersection = pred_elements.intersection(gold_elements)

    recall = len(intersection) / len(gold_elements) if gold_elements else 0.0
    precision = len(intersection) / len(pred_elements) if pred_elements else 0.0
    
    missing = list(gold_elements - pred_elements)
    extra = list(pred_elements - gold_elements)
    
    return recall, precision, missing, extra

In [4]:
import os
import json
import sqlite3
import pandas as pd
from tqdm import tqdm

In [14]:
def get_db_schema(db_path: str):
    """SQLite DB 파일에서 모든 테이블과 컬럼 정보를 추출합니다."""
    conn = sqlite3.connect(db_path)
    conn.text_factory = lambda b: b.decode(errors='ignore')
    cursor = conn.cursor()
    
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0].lower() for row in cursor.fetchall() if row[0].lower() != 'sqlite_sequence']
    
    schema = {}
    for table in tables:
        cursor.execute(f"PRAGMA table_info(`{table}`)")
        cols = [row[1].lower() for row in cursor.fetchall()]
        schema[table] = cols
        
    conn.close()
    return schema

def calculate_theoretical_bounds():
    dataset_path = '/home/hyeonjin/thesis_refactored/data/raw/BIRD_dev/dev.json'
    db_base_dir = '/home/hyeonjin/thesis_refactored/data/raw/BIRD_dev/dev_databases'
    
    with open(dataset_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
        
    # 결과 저장용 딕셔너리에 전체(Overall) 지표 추가
    results = {
        'full_schema': {'o_rec': [], 'o_prec': [], 't_rec': [], 't_prec': [], 'c_rec': [], 'c_prec': []},
        'gold_table_schema': {'o_rec': [], 'o_prec': [], 't_rec': [], 't_prec': [], 'c_rec': [], 'c_prec': []},
        'gold_column_schema': {'o_rec': [], 'o_prec': [], 't_rec': [], 't_prec': [], 'c_rec': [], 'c_prec': []}
    }
    
    for item in tqdm(dataset, desc="Calculating Theoretical Metrics"):
        db_id = item['db_id']
        gold_sql = item.get('SQL', item.get('query', ''))
        db_path = os.path.join(db_base_dir, db_id, f"{db_id}.sqlite")
        
        if not os.path.exists(db_path):
            continue
            
        # 1. Gold 정답 추출
        gold_tables, gold_cols = parse_sql_elements(gold_sql)
        gold_all = gold_tables.union(gold_cols)  # 💡 전체 정답 집합
        
        db_schema = get_db_schema(db_path)
        
        # ==========================================
        # Case 1: Full Schema (데이터베이스 전체)
        # ==========================================
        full_tables = set(db_schema.keys())
        # 💡 [핵심 수정] evaluator.py와 동일하게 순수 컬럼명만 추출하도록 수정!
        full_cols = set([c for t, cols in db_schema.items() for c in cols])
        full_all = full_tables.union(full_cols) 
        
        o_rec, o_prec, _, _ = calculate_schema_metrics(full_all, gold_all)
        t_rec, t_prec, _, _ = calculate_schema_metrics(full_tables, gold_tables)
        c_rec, c_prec, _, _ = calculate_schema_metrics(full_cols, gold_cols)
        
        results['full_schema']['o_rec'].append(o_rec)
        results['full_schema']['o_prec'].append(o_prec)
        results['full_schema']['t_rec'].append(t_rec)
        results['full_schema']['t_prec'].append(t_prec)
        results['full_schema']['c_rec'].append(c_rec)
        results['full_schema']['c_prec'].append(c_prec)
        
        # ==========================================
        # Case 2: Gold Table Schema (정답 테이블의 모든 컬럼)
        # ==========================================
        gt_tables = gold_tables
        # 💡 [핵심 수정] 순수 컬럼명 추출
        gt_cols = set([c for t in gold_tables if t in db_schema for c in db_schema[t]])
        gt_all = gt_tables.union(gt_cols) 
        
        o_rec, o_prec, _, _ = calculate_schema_metrics(gt_all, gold_all)
        t_rec, t_prec, _, _ = calculate_schema_metrics(gt_tables, gold_tables)
        c_rec, c_prec, _, _ = calculate_schema_metrics(gt_cols, gold_cols)
        
        results['gold_table_schema']['o_rec'].append(o_rec)
        results['gold_table_schema']['o_prec'].append(o_prec)
        results['gold_table_schema']['t_rec'].append(t_rec)
        results['gold_table_schema']['t_prec'].append(t_prec)
        results['gold_table_schema']['c_rec'].append(c_rec)
        results['gold_table_schema']['c_prec'].append(c_prec)
        
        # ==========================================
        # Case 3: Gold Column Schema (정답 테이블 + 정답 컬럼)
        # ==========================================
        gc_all = gold_tables.union(gold_cols)
        
        o_rec, o_prec, _, _ = calculate_schema_metrics(gc_all, gold_all)
        t_rec, t_prec, _, _ = calculate_schema_metrics(gold_tables, gold_tables)
        c_rec, c_prec, _, _ = calculate_schema_metrics(gold_cols, gold_cols)
        
        results['gold_column_schema']['o_rec'].append(o_rec)
        results['gold_column_schema']['o_prec'].append(o_prec)
        results['gold_column_schema']['t_rec'].append(t_rec)
        results['gold_column_schema']['t_prec'].append(t_prec)
        results['gold_column_schema']['c_rec'].append(c_rec)
        results['gold_column_schema']['c_prec'].append(c_prec)

    # 2. 평균 계산 및 DataFrame 출력
    print("\n" + "="*80)
    print("📊 Theoretical Schema Linking Bounds (with Overall Metrics)")
    print("="*80)
    
    summary = []
    for case_name, metrics in results.items():
        avg_metrics = {k: sum(v)/len(v) if v else 0.0 for k, v in metrics.items()}
        
        summary.append({
            "Scenario": case_name,
            "Overall Recall": round(avg_metrics['o_rec'], 4),
            "Overall Precision": round(avg_metrics['o_prec'], 4),
            "Table Recall": round(avg_metrics['t_rec'], 4),
            "Table Precision": round(avg_metrics['t_prec'], 4),
            "Col Recall": round(avg_metrics['c_rec'], 4),
            "Col Precision": round(avg_metrics['c_prec'], 4)
        })
        
    df = pd.DataFrame(summary)
    print(df.to_string(index=False))
    print("="*80)

In [15]:
calculate_theoretical_bounds()

Calculating Theoretical Metrics:   0%|          | 0/1534 [00:00<?, ?it/s]

Calculating Theoretical Metrics: 100%|██████████| 1534/1534 [00:01<00:00, 866.38it/s]


📊 Theoretical Schema Linking Bounds (with Overall Metrics)
          Scenario  Overall Recall  Overall Precision  Table Recall  Table Precision  Col Recall  Col Precision
       full_schema          0.9969             0.1381        0.9979           0.3292      0.9968         0.1173
 gold_table_schema          0.9978             0.3359        1.0000           1.0000      0.9968         0.2729
gold_column_schema          1.0000             1.0000        1.0000           1.0000      1.0000         1.0000
